# 4. An amortized operator: the model writes the heuristic

**What you get out of this notebook:** the same loop searching program space
instead of solution space — the artifact is a `priority(item, bins)` function for
online bin packing — and what changes in the validator when the payload is code.

*Amortized* means the model is paid once, offline. The emitted heuristic then runs
on new instances with no further model calls.

In [1]:
import _bootstrap

import bpp_amortized as bpp
from llm import load_pool

## What the operator is asked for

In [2]:
print(bpp.Spec().render(None, 22, []))

[context] online bin packing, capacity 10; artifact: a Python priority(item, bins) returning one score per open bin.
[conditioning] current best total bins = 22.
[instruction] Emit one improved priority function.
[format] CANDIDATE envelope with code in the payload.


## What it proposes

Three completions, in this order: one that is not a valid heuristic at all, one
worst-fit, one best-fit. They are in `fixtures/bpp_pool.txt`, which both this
notebook and the script read.

In [3]:
for cand in load_pool('bpp_pool.txt'):
    print(cand.split('payload:')[1].split('END_CANDIDATE')[0].strip())
    print('-' * 60)

def priority(item, bins):
    return 0   # invalid: not one score per bin
------------------------------------------------------------
def priority(item, bins):
    return [b - item for b in bins]
------------------------------------------------------------
def priority(item, bins):
    return [-(b - item) for b in bins]
------------------------------------------------------------


## The validator has more work here

With code as the payload there are two extra questions: is it parseable, and does
it do what the interface promises? The second one is answered by running it — on a
probe input, before spending a full evaluation on it.

In [4]:
scalar = 'def priority(item, bins):\n    return 0'
from llm import envelope
fn = bpp.parse(envelope(scalar))
try:
    bpp.feasible(fn)
except ValueError as err:
    print(err)

feasibility: priority must return one score per bin


Note also what `parse` refuses before executing anything: an `import` in the
payload. That check, plus a restricted set of builtins, is a weak sandbox and the
code says so. It is enough for a fixed pool of completions; it is not enough for a
live model.

## The run

In [5]:
best, best_score = bpp.main()

lower bound (total bins) = 19
first-fit start: total bins = 22  (gap 15.8%)
  step 0: rejected 22
  step 1: accepted 19
  step 2: rejected 22
  step 3: rejected 19
best heuristic: total bins = 19  (gap 0.0%)


First fit needs 22 bins on these instances and the lower bound is 19. The operator
reaches 19: best fit, the known good rule for this problem. The step that was
refused never reached the evaluator.

Next: [5. Putting the framework to the test](05_drop_channel_ablation.ipynb).